In [ ]:
from google.colab import userdata

TMDB_API_KEY = userdata.get("TMDB_API_KEY")

print("TMDB API key loaded successfully!")

TMDB API key loaded successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving movies_final.csv to movies_final.csv


In [ ]:
import pandas as pd
import numpy as np
import requests
import time

movies = pd.read_csv("movies_final.csv")
links = pd.read_csv("links.csv")

print("Movies shape:", movies.shape)
print("Links shape:", links.shape)

display(movies.head())
display(links.head())

Movies shape: (9742, 9)
Links shape: (9742, 3)


,movieId,title,genres,year,clean_title,genre_list,combined_tags,rating_count,average_rating
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,"['Adventure', 'Animation', 'Children', 'Comedy...",pixar pixar fun,215,3.92
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,"['Adventure', 'Children', 'Fantasy']",fantasy magic board game robin williams game,110,3.43
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,"['Comedy', 'Romance']",moldy old,52,3.26
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,"['Comedy', 'Drama', 'Romance']",NaN,7,2.36
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,['Comedy'],pregnancy remake,49,3.07


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [ ]:
movies_tmdb = movies.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="left"
)

print("Merged dataset shape:", movies_tmdb.shape)

print(
    "Missing TMDB IDs:",
    movies_tmdb["tmdbId"].isna().sum()
)

display(
    movies_tmdb[
        ["movieId", "clean_title", "year", "tmdbId"]
    ].head(10)
)

Merged dataset shape: (9742, 10)
Missing TMDB IDs: 8


,movieId,clean_title,year,tmdbId
0,1,Toy Story,1995.0,862.0
1,2,Jumanji,1995.0,8844.0
2,3,Grumpier Old Men,1995.0,15602.0
3,4,Waiting to Exhale,1995.0,31357.0
4,5,Father of the Bride Part II,1995.0,11862.0
5,6,Heat,1995.0,949.0
6,7,Sabrina,1995.0,11860.0
7,8,Tom and Huck,1995.0,45325.0
8,9,Sudden Death,1995.0,9091.0
9,10,GoldenEye,1995.0,710.0


In [ ]:
# Test TMDB API using Toy Story's TMDB ID
test_tmdb_id = 862

url = f"https://api.themoviedb.org/3/movie/{test_tmdb_id}"

params = {
    "api_key": TMDB_API_KEY,
    "language": "en-US"
}

response = requests.get(url, params=params)

print("Status Code:", response.status_code)

if response.status_code == 200:
    movie_data = response.json()

    print("API connection successful!")
    print("Title:", movie_data.get("title"))
    print("Runtime:", movie_data.get("runtime"))
    print("Release Date:", movie_data.get("release_date"))
    print("Popularity:", movie_data.get("popularity"))
    print("Overview:", movie_data.get("overview"))
else:
    print("API request failed.")
    print(response.text)

Status Code: 200
API connection successful!
Title: Toy Story
Runtime: 81
Release Date: 1995-11-22
Popularity: 30.9498
Overview: Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.


## Step 4: TMDB Metadata Fetching

This section enriches the MovieLens dataset with additional movie metadata retrieved from the TMDB API using the TMDB IDs available in `links.csv`.

The metadata includes runtime, overview, popularity, release date, poster information, language, status, tagline, and other useful attributes for building the context-aware recommendation system.

In [ ]:
def fetch_tmdb_metadata(tmdb_id):
    """
    Fetch metadata for a single movie from the TMDB API.
    """

    # Handle missing TMDB IDs
    if pd.isna(tmdb_id):
        return None

    try:
        tmdb_id = int(tmdb_id)

        url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"

        params = {
            "api_key": TMDB_API_KEY,
            "language": "en-US"
        }

        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        if response.status_code == 200:

            data = response.json()

            # Extract genre names
            genres = [
                genre["name"]
                for genre in data.get("genres", [])
            ]

            return {
                "tmdbId": tmdb_id,
                "tmdb_title": data.get("title"),
                "original_title": data.get("original_title"),
                "overview": data.get("overview"),
                "runtime": data.get("runtime"),
                "release_date": data.get("release_date"),
                "tmdb_popularity": data.get("popularity"),
                "vote_average": data.get("vote_average"),
                "vote_count": data.get("vote_count"),
                "original_language": data.get("original_language"),
                "poster_path": data.get("poster_path"),
                "backdrop_path": data.get("backdrop_path"),
                "adult": data.get("adult"),
                "status": data.get("status"),
                "tagline": data.get("tagline"),
                "tmdb_genres": "|".join(genres)
            }

        elif response.status_code == 404:
            print(f"Movie not found: TMDB ID {tmdb_id}")
            return None

        else:
            print(
                f"Request failed for TMDB ID {tmdb_id}. "
                f"Status: {response.status_code}"
            )
            return None

    except requests.exceptions.RequestException as e:

        print(
            f"Network error for TMDB ID {tmdb_id}: {e}"
        )

        return None

    except Exception as e:

        print(
            f"Error processing TMDB ID {tmdb_id}: {e}"
        )

        return None

In [ ]:
test_metadata = fetch_tmdb_metadata(862)

test_metadata

{'tmdbId': 862,
 'tmdb_title': 'Toy Story',
 'original_title': 'Toy Story',
 'overview': "Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.",
 'runtime': 81,
 'release_date': '1995-11-22',
 'tmdb_popularity': 30.9498,
 'vote_average': 7.983,
 'vote_count': 20174,
 'original_language': 'en',
 'poster_path': '/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg',
 'backdrop_path': '/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg',
 'adult': False,
 'status': 'Released',
 'tagline': 'The adventure takes off when toys come to life!',
 'tmdb_genres': 'Family|Comedy|Animation|Adventure'}

In [ ]:
pd.DataFrame(
    [test_metadata]
).T

,0
tmdbId,862
tmdb_title,Toy Story
original_title,Toy Story
overview,"Led by Woody, Andy's toys live happily in his ..."
runtime,81
release_date,1995-11-22
tmdb_popularity,30.9498
vote_average,7.983
vote_count,20174
original_language,en


In [ ]:
tenet_metadata = fetch_tmdb_metadata(577922)

pd.DataFrame([tenet_metadata]).T

,0
tmdbId,577922
tmdb_title,Tenet
original_title,Tenet
overview,Armed with only one word - Tenet - and fightin...
runtime,150
release_date,2020-08-22
tmdb_popularity,15.2245
vote_average,7.176
vote_count,11229
original_language,en


In [ ]:
movies_to_fetch = movies_tmdb[
    movies_tmdb["tmdbId"].notna()
].copy()

print("Total movies:", len(movies_tmdb))
print("Movies with valid TMDB IDs:", len(movies_to_fetch))
print("Movies without TMDB IDs:", movies_tmdb["tmdbId"].isna().sum())

Total movies: 9742
Movies with valid TMDB IDs: 9734
Movies without TMDB IDs: 8


In [ ]:
from tqdm.auto import tqdm
import os

checkpoint_file = "tmdb_metadata_checkpoint.csv"

# Resume from checkpoint if it exists
if os.path.exists(checkpoint_file):

    metadata_df = pd.read_csv(checkpoint_file)

    fetched_ids = set(
        metadata_df["tmdbId"]
        .dropna()
        .astype(int)
    )

    metadata_records = metadata_df.to_dict("records")

    print(
        f"Checkpoint found. "
        f"{len(fetched_ids)} movies already processed."
    )

else:

    fetched_ids = set()
    metadata_records = []

    print("No checkpoint found. Starting from the beginning.")


# Get movies that still need to be fetched
remaining_movies = movies_to_fetch[
    ~movies_to_fetch["tmdbId"]
    .astype(int)
    .isin(fetched_ids)
]

print(
    "Movies remaining:",
    len(remaining_movies)
)

No checkpoint found. Starting from the beginning.
Movies remaining: 9734


In [ ]:
for count, (_, row) in enumerate(
    tqdm(
        remaining_movies.iterrows(),
        total=len(remaining_movies),
        desc="Fetching TMDB metadata"
    ),
    start=1
):

    tmdb_id = row["tmdbId"]

    metadata = fetch_tmdb_metadata(
        tmdb_id
    )

    if metadata is not None:
        metadata_records.append(metadata)

    # Save checkpoint every 100 requests
    if count % 100 == 0:

        pd.DataFrame(
            metadata_records
        ).to_csv(
            checkpoint_file,
            index=False
        )

# Final checkpoint
metadata_df = pd.DataFrame(
    metadata_records
)

metadata_df.to_csv(
    checkpoint_file,
    index=False
)

print("\nTMDB metadata fetching completed!")

print(
    "Metadata records collected:",
    len(metadata_df)
)

Fetching TMDB metadata:   0%|          | 0/9734 [00:00<?, ?it/s]

Movie not found: TMDB ID 12773
Movie not found: TMDB ID 17882
Movie not found: TMDB ID 68149
Movie not found: TMDB ID 24549
Movie not found: TMDB ID 14980
Movie not found: TMDB ID 164721
Movie not found: TMDB ID 140207
Movie not found: TMDB ID 192936
Movie not found: TMDB ID 876
Movie not found: TMDB ID 149645
Movie not found: TMDB ID 8677
Movie not found: TMDB ID 13057
Movie not found: TMDB ID 2670
Movie not found: TMDB ID 215993
Movie not found: TMDB ID 13519
Movie not found: TMDB ID 152426
Movie not found: TMDB ID 30983
Movie not found: TMDB ID 7096
Movie not found: TMDB ID 110147
Network error for TMDB ID 437: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
Movie not found: TMDB ID 206216
Movie not found: TMDB ID 19341
Movie not found: TMDB ID 36763
Movie not found: TMDB ID 13716
Movie not found: TMDB ID 10700
Movie not found: TMDB ID 58923
Movie not found: TMDB ID 17266
Network error for TMDB ID 7485: ('Connection aborted.', ConnectionResetError(104,

In [ ]:
print(
    "Metadata shape:",
    metadata_df.shape
)

print("\nMissing values:")
print(
    metadata_df.isnull().sum()
)

display(
    metadata_df.head()
)

Metadata shape: (9615, 16)

Missing values:
tmdbId                0
tmdb_title            0
original_title        0
overview              0
runtime               0
release_date          0
tmdb_popularity       0
vote_average          0
vote_count            0
original_language     0
poster_path           3
backdrop_path        78
adult                 0
status                0
tagline               0
tmdb_genres           0
dtype: int64


,tmdbId,tmdb_title,original_title,overview,runtime,release_date,tmdb_popularity,vote_average,vote_count,original_language,poster_path,backdrop_path,adult,status,tagline,tmdb_genres
0,862,Toy Story,Toy Story,"Led by Woody, Andy's toys live happily in his ...",81,1995-11-22,30.9498,7.983,20174,en,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg,False,Released,The adventure takes off when toys come to life!,Family|Comedy|Animation|Adventure
1,8844,Jumanji,Jumanji,When siblings Judy and Peter discover an encha...,104,1995-12-15,2.4359,7.249,11395,en,/iWV47r6kFneCiApgrMII5HSkfHw.jpg,/qSxeCfWUUyht9hZgaaYmtPtTkw2.jpg,False,Released,It's a jungle in here.,Adventure|Fantasy|Family
2,15602,Grumpier Old Men,Grumpier Old Men,A family wedding reignites the ancient feud be...,101,1995-12-22,1.6558,6.479,432,en,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,/1o4vuCHpmd4DXofMYDUwpnhKiuy.jpg,False,Released,Still Yelling. Still Fighting. Still Ready for...,Romance|Comedy
3,31357,Waiting to Exhale,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",127,1995-12-22,1.8246,6.261,207,en,/4wjGMwPsdlvi025ZqR4rXnFDvBz.jpg,/jZjoEKXMTDoZAGdkjhAdJaKtXSN.jpg,False,Released,Friends are the people who let you be yourself...,Comedy|Drama|Romance
4,11862,Father of the Bride Part II,Father of the Bride Part II,Just when George Banks has recovered from his ...,106,1995-12-08,2.1537,6.271,819,en,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,/lEsjVrGU21BeJjF5AF9EWsihDpw.jpg,False,Released,Just when his world is back to normal... he's ...,Comedy|Family


In [ ]:
metadata_df.to_csv(
    "tmdb_metadata.csv",
    index=False
)

print(
    "Saved successfully as tmdb_metadata.csv"
)

Saved successfully as tmdb_metadata.csv


In [ ]:
# Step 5.6 - Merge MovieLens data with TMDB metadata

# Ensure tmdbId has the same data type in both datasets
movies_tmdb["tmdbId"] = movies_tmdb["tmdbId"].astype("Int64")
metadata_df["tmdbId"] = metadata_df["tmdbId"].astype("Int64")

# Merge the MovieLens dataset with TMDB metadata
movies_enriched = movies_tmdb.merge(
    metadata_df,
    on="tmdbId",
    how="left"
)

print("Enriched dataset shape:", movies_enriched.shape)

print(
    "Movies with TMDB metadata:",
    movies_enriched["tmdb_title"].notna().sum()
)

print(
    "Movies without TMDB metadata:",
    movies_enriched["tmdb_title"].isna().sum()
)

# Preview selected columns
display(
    movies_enriched[
        [
            "movieId",
            "clean_title",
            "tmdbId",
            "tmdb_title",
            "runtime",
            "release_date",
            "tmdb_popularity",
            "vote_average",
            "original_language"
        ]
    ].head(10)
)

Enriched dataset shape: (9744, 25)
Movies with TMDB metadata: 9617
Movies without TMDB metadata: 127


,movieId,clean_title,tmdbId,tmdb_title,runtime,release_date,tmdb_popularity,vote_average,original_language
0,1,Toy Story,862,Toy Story,81.0,1995-11-22,30.9498,7.983,en
1,2,Jumanji,8844,Jumanji,104.0,1995-12-15,2.4359,7.249,en
2,3,Grumpier Old Men,15602,Grumpier Old Men,101.0,1995-12-22,1.6558,6.479,en
3,4,Waiting to Exhale,31357,Waiting to Exhale,127.0,1995-12-22,1.8246,6.261,en
4,5,Father of the Bride Part II,11862,Father of the Bride Part II,106.0,1995-12-08,2.1537,6.271,en
5,6,Heat,949,Heat,170.0,1995-12-15,11.9265,7.900,en
6,7,Sabrina,11860,Sabrina,127.0,1995-12-15,2.6983,6.212,en
7,8,Tom and Huck,45325,Tom and Huck,97.0,1995-12-22,0.9201,5.302,en
8,9,Sudden Death,9091,Sudden Death,111.0,1995-10-27,1.6499,6.028,en
9,10,GoldenEye,710,GoldenEye,130.0,1995-11-16,5.0216,6.905,en


In [ ]:
print("Duplicate TMDB IDs in metadata:",
      metadata_df["tmdbId"].duplicated().sum())

duplicates = metadata_df[
    metadata_df["tmdbId"].duplicated(keep=False)
].sort_values("tmdbId")

display(duplicates)

Duplicate TMDB IDs in metadata: 1


,tmdbId,tmdb_title,original_title,overview,runtime,release_date,tmdb_popularity,vote_average,vote_count,original_language,poster_path,backdrop_path,adult,status,tagline,tmdb_genres
4161,4912,Confessions of a Dangerous Mind,Confessions of a Dangerous Mind,"Television made him famous, but his biggest hi...",113,2002-12-31,1.8997,6.689,1276,en,/mM58ABMRzwn5zVx6BS7IiNK0j3F.jpg,/uS03jRk64ZFFtpKNAhLppXa1nKP.jpg,False,Released,Some things are better left top secret.,Comedy|Crime|Drama|Romance|Thriller|History
9008,4912,Confessions of a Dangerous Mind,Confessions of a Dangerous Mind,"Television made him famous, but his biggest hi...",113,2002-12-31,1.8997,6.689,1276,en,/mM58ABMRzwn5zVx6BS7IiNK0j3F.jpg,/uS03jRk64ZFFtpKNAhLppXa1nKP.jpg,False,Released,Some things are better left top secret.,Comedy|Crime|Drama|Romance|Thriller|History


In [ ]:
# Remove duplicate TMDB records
metadata_df = metadata_df.drop_duplicates(
    subset=["tmdbId"],
    keep="first"
)

print("Metadata shape after removing duplicates:", metadata_df.shape)
print("Remaining duplicate TMDB IDs:", metadata_df["tmdbId"].duplicated().sum())

# Re-merge
movies_enriched = movies_tmdb.merge(
    metadata_df,
    on="tmdbId",
    how="left"
)

print("Final enriched dataset shape:", movies_enriched.shape)
print("Movies with TMDB metadata:", movies_enriched["tmdb_title"].notna().sum())
print("Movies without TMDB metadata:", movies_enriched["tmdb_title"].isna().sum())

Metadata shape after removing duplicates: (9614, 16)
Remaining duplicate TMDB IDs: 0
Final enriched dataset shape: (9742, 25)
Movies with TMDB metadata: 9615
Movies without TMDB metadata: 127


In [ ]:
# Step 5.8 - Final validation of enriched dataset

print("=" * 50)
print("FINAL ENRICHED DATASET VALIDATION")
print("=" * 50)

print("\nDataset shape:")
print(movies_enriched.shape)

print("\nDuplicate movie IDs:")
print(movies_enriched["movieId"].duplicated().sum())

print("\nMissing values in important TMDB columns:")
important_columns = [
    "tmdb_title",
    "overview",
    "runtime",
    "release_date",
    "original_language",
    "poster_path"
]

print(movies_enriched[important_columns].isnull().sum())

print("\nSample enriched movies:")

display(
    movies_enriched[
        [
            "movieId",
            "clean_title",
            "genres",
            "average_rating",
            "rating_count",
            "tmdb_title",
            "runtime",
            "release_date",
            "tmdb_popularity",
            "vote_average",
            "original_language"
        ]
    ].head(10)
)

FINAL ENRICHED DATASET VALIDATION

Dataset shape:
(9742, 25)

Duplicate movie IDs:
0

Missing values in important TMDB columns:
tmdb_title           127
overview             127
runtime              127
release_date         127
original_language    127
poster_path          130
dtype: int64

Sample enriched movies:


,movieId,clean_title,genres,average_rating,rating_count,tmdb_title,runtime,release_date,tmdb_popularity,vote_average,original_language
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,3.92,215,Toy Story,81.0,1995-11-22,30.9498,7.983,en
1,2,Jumanji,Adventure|Children|Fantasy,3.43,110,Jumanji,104.0,1995-12-15,2.4359,7.249,en
2,3,Grumpier Old Men,Comedy|Romance,3.26,52,Grumpier Old Men,101.0,1995-12-22,1.6558,6.479,en
3,4,Waiting to Exhale,Comedy|Drama|Romance,2.36,7,Waiting to Exhale,127.0,1995-12-22,1.8246,6.261,en
4,5,Father of the Bride Part II,Comedy,3.07,49,Father of the Bride Part II,106.0,1995-12-08,2.1537,6.271,en
5,6,Heat,Action|Crime|Thriller,3.95,102,Heat,170.0,1995-12-15,11.9265,7.900,en
6,7,Sabrina,Comedy|Romance,3.19,54,Sabrina,127.0,1995-12-15,2.6983,6.212,en
7,8,Tom and Huck,Adventure|Children,2.88,8,Tom and Huck,97.0,1995-12-22,0.9201,5.302,en
8,9,Sudden Death,Action,3.12,16,Sudden Death,111.0,1995-10-27,1.6499,6.028,en
9,10,GoldenEye,Action|Adventure|Thriller,3.50,132,GoldenEye,130.0,1995-11-16,5.0216,6.905,en


In [ ]:
# Step 5.9 - Save final enriched movie dataset

movies_enriched.to_csv(
    "movies_enriched.csv",
    index=False
)

print("Saved successfully as movies_enriched.csv")
print("Final dataset shape:", movies_enriched.shape)

Saved successfully as movies_enriched.csv
Final dataset shape: (9742, 25)
